In [2]:
import torch
import torchvision
from torchvision.transforms import v2
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader



In [4]:
train_dataset = torchvision.datasets.CIFAR10(root='/data',
                                             train=True,
                                             download=True,
                                             transform=v2.Compose([v2.ToImage(),
                                                                   v2.ToDtype(torch.float32,scale=True),
                                                                   v2.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])]))
test_dataset = torchvision.datasets.CIFAR10(root='/data',
                                             train=False,
                                             download=True,
                                             transform=v2.Compose([v2.ToImage(),
                                                                   v2.ToDtype(torch.float32,scale=True),
                                                                   v2.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])]))


100%|██████████| 170M/170M [20:26<00:00, 139kB/s]


In [5]:
tasks=[[0,1],[2],[3],[4],[5],[6],[7],[8],[9]]

In [6]:
from torch.utils.data import Subset
task_datasets=[]

for classes in tasks:
  indices=[i for i ,(_,label) in enumerate(train_dataset) if label in classes]
  task_datasets.append(Subset(train_dataset,indices))

task_test_dataset=[]
for classes in tasks:
  indices=[i for i, (_,label) in enumerate(test_dataset) if label in classes]
  task_test_dataset.append(Subset(test_dataset,indices))

In [7]:
task1_loader=DataLoader(task_datasets[0],batch_size=10,shuffle=True)
test1_loader=DataLoader(task_test_dataset[0],batch_size=10,shuffle=True)

In [ ]:
class ImageNet(nn.Module):

  def __init__(self):
    super().__init__()
    self.conv1=nn.Conv2d(3,5, 5, padding=2)
    self.conv2=nn.Conv2d(5,10, 5, padding=2)
    self.conv3=nn.Conv2d(10,20, 5, padding=2)
    self.conv4=nn.Conv2d(20,30, 5, padding=2)
    self.pool=nn.MaxPool2d(2,2)
    self.fc1=nn.Linear(30*2*2,80)
    self.fc2=nn.Linear(80,40)
    self.fc3=nn.Linear(40,10)
    self.fc4=nn.Linear(10,2)
  def forward(self,x):
    x=self.pool(F.relu(self.conv1(x)))
    x=self.pool(F.relu(self.conv2(x)))
    x=self.pool(F.relu(self.conv3(x)))
    x=self.pool(F.relu(self.conv4(x)))
    x=torch.flatten(x,1)
    x=F.relu(self.fc1(x))
    x=F.relu(self.fc2(x))
    x=F.relu(self.fc3(x))
    x=self.fc4(x)
    return x

In [13]:
net=ImageNet()
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(net.parameters(),lr=0.001,momentum=0.9)

In [ ]:
result_loss=0
for epoch in range(10):
  for batch,data in enumerate(task1_loader):
    image,labels=data
    output=net(image)
    optimizer.zero_grad()
    loss=criterion(output,labels)
    loss.backward()
    optimizer.step()

    result_loss+=loss.item()

    if batch%100==99:
      print(f"Epoch:{epoch+1} , Batch:{batch+1} , Loss:{result_loss/1000:.4f}")
      result_loss=0

Epoch:1 , Batch:100 , Loss:0.06932780092954635
Epoch:1 , Batch:200 , Loss:0.06926117378473282
Epoch:1 , Batch:300 , Loss:0.06928764235973359
Epoch:1 , Batch:400 , Loss:0.06937045139074326
Epoch:1 , Batch:500 , Loss:0.06931797713041306
Epoch:1 , Batch:600 , Loss:0.06930889564752579
Epoch:1 , Batch:700 , Loss:0.06931567311286926
Epoch:1 , Batch:800 , Loss:0.06931137681007385
Epoch:1 , Batch:900 , Loss:0.06931736469268798
Epoch:1 , Batch:1000 , Loss:0.06930351376533508
Epoch:2 , Batch:100 , Loss:0.0693174963593483
Epoch:2 , Batch:200 , Loss:0.06929497182369232
Epoch:2 , Batch:300 , Loss:0.0693160766363144
Epoch:2 , Batch:400 , Loss:0.06928450894355774
Epoch:2 , Batch:500 , Loss:0.06931668519973755
Epoch:2 , Batch:600 , Loss:0.06930548757314682
Epoch:2 , Batch:700 , Loss:0.06930063843727112
Epoch:2 , Batch:800 , Loss:0.06928950595855712
Epoch:2 , Batch:900 , Loss:0.0693045859336853
Epoch:2 , Batch:1000 , Loss:0.06929325240850448
Epoch:3 , Batch:100 , Loss:0.06929265558719636
Epoch:3 , Batc

In [ ]:
correct_label=0
total_label=0

net.eval()
with torch.no_grad():
  for image,label in test1_loader:
    output=net(image)
    _,predicted=torch.max(output,1)
    total_label+=label.size(0)
    correct_label+=(predicted==label).sum().item()

In [ ]:
print(f"Accuracy is {100*correct_label/total_label:.2f}")

Accuracy is 92.65
